# 08 端到端实战：Kaggle 工作流

> 前置：本模块 01-07 全部。
> 目标：把前面七课串成一条**完整生产线**——EDA → 特征工程 → 基线 → 调参 → 集成 → 提交文件。做完这一课，你就具备独立打一场 Kaggle 竞赛（或独立交付一个模型项目）的完整能力。

## Kaggle 工作流（刻进脑子）

```
读题（指标是什么！）→ EDA → 特征工程 → 基线 → 交叉验证 → 调参
   → 集成/精修 → 生成提交文件 → 验证提交 → 复盘（错误分析 → 回炉）
```

**第一原则：先搞清评估指标**。Titanic 用准确率、House Prices 用 RMSE、Porto Seguro 用 Gini——指标决定你怎么调模型（05 课）。

In [ ]:
# 本模块通用导入（全部 CPU 即可运行）
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import warnings
warnings.filterwarnings("ignore")

# 中文字体兼容（Windows / macOS）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

In [ ]:
TITANIC_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
# 真实竞赛是 train.csv（含标签）+ test.csv（无标签）；镜像只有一份数据，
# 本课自行切分模拟 Kaggle 的 test 集（方法与流程完全一致）。
df = pd.read_csv(TITANIC_URL).set_index("PassengerId")
print("Kaggle 风格：train 含标签 / test 无标签。本课用镜像数据自行切分模拟。")
print("数据形状：", df.shape)

# ---- EDA 1：缺失 ----
print("\n缺失值：")
print(df.isna().sum()[df.isna().sum() > 0])

# ---- EDA 2：目标分布 ----
print("\n目标分布：", df["Survived"].value_counts(normalize=True).round(3).to_dict())

# ---- EDA 3：连续特征 ----
print("\nAge / Fare 统计：")
print(df[["Age", "Fare"]].describe().round(2).to_string())

# ---- EDA 4：数值特征与目标的相关性 ----
print("\n数值特征与 Survived 的相关系数：")
print(df.select_dtypes("number").corr()["Survived"].round(3).sort_values(ascending=False).to_string())

# ---- EDA 5：类别特征 vs 生存率 ----
print("\n性别生存率：", df.groupby("Sex")["Survived"].mean().round(3).to_dict())
print("舱位生存率：", df.groupby("Pclass")["Survived"].mean().round(3).to_dict())

In [ ]:
# ---- 特征工程（完整版）：从原始字段挖出模型能用的信息 ----
def engineer(df):
    d = df.copy()
    # 1) Title：从姓名提取称呼（最有信息量的文本特征）
    d["Title"] = d["Name"].str.extract(r" ([A-Za-z]+)\.")[0]
    rare = ["Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"]
    d["Title"] = d["Title"].replace(rare, "Rare")
    d["Title"] = d["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    # 2) Age：按 Title 分组中位数填充（比全局中位数更细）
    d["Age"] = d["Age"].fillna(d.groupby("Title")["Age"].transform("median"))
    d["Age"] = d["Age"].fillna(d["Age"].median())
    d["Fare"] = d["Fare"].fillna(d["Fare"].median())
    d["Embarked"] = d["Embarked"].fillna(d["Embarked"].mode()[0])
    # 3) 家庭结构
    d["FamilySize"] = d["SibSp"] + d["Parch"] + 1
    d["IsAlone"] = (d["FamilySize"] == 1).astype(int)
    d["HasCabin"] = d["Cabin"].notna().astype(int)
    # 4) 年龄分箱
    d["AgeBin"] = pd.cut(d["Age"], [0, 12, 20, 40, 60, 100], labels=False)
    # 5) 类别编码
    d = pd.get_dummies(d, columns=["Sex", "Embarked", "Title"], drop_first=True)
    return d

train = engineer(df)
y = train["Survived"].astype(int)
drop_cols = ["Survived", "Name", "Ticket", "Cabin"]
X = train.drop(columns=[c for c in drop_cols if c in train.columns])
print("特征数：", X.shape[1])
print("特征列表：", list(X.columns))

In [ ]:
# ---- 基线 + 5 折交叉验证 ----
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

cv = StratifiedKFold(5, shuffle=True, random_state=0)
models = {
    "逻辑回归": LogisticRegression(max_iter=1000),
    "随机森林": RandomForestClassifier(n_estimators=200, random_state=0),
    "GBDT": HistGradientBoostingClassifier(max_iter=200, random_state=0),
}
print(f"{'模型':12s} {'CV-AUC':>8s} {'CV-准确率':>10s}")
for name, m in models.items():
    auc = cross_val_score(m, X, y, cv=cv, scoring="roc_auc").mean()
    acc = cross_val_score(m, X, y, cv=cv, scoring="accuracy").mean()
    print(f"{name:12s} {auc:8.4f} {acc:10.4f}")

In [ ]:
# ---- 调参：随机搜索（04 课方法）----
param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 5, 8, 12],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}
rs = RandomizedSearchCV(RandomForestClassifier(random_state=0), param_grid,
                        n_iter=15, cv=cv, scoring="roc_auc", random_state=0, n_jobs=-1)
rs.fit(X, y)
print("最优超参：", rs.best_params_)
print("最优 CV-AUC：", round(rs.best_score_, 4))

## 集成：最后一搏

单个模型有各自的偏差，**把差异化的模型平均**通常再涨 1~2 个点：

- **Voting（软投票）**：多个模型预测概率取平均；
- **Stacking**：用第一层模型的预测做第二层模型的输入（更高阶）。

原则：集成里模型的**多样性**比数量重要——一个 GBDT + 一个随机森林 + 一个逻辑回归，比三个随机森林更有用。

In [ ]:
# 集成：软投票
from sklearn.ensemble import VotingClassifier
vote = VotingClassifier([
    ("gbdt", HistGradientBoostingClassifier(max_iter=200, random_state=0)),
    ("rf", RandomForestClassifier(**rs.best_params_, random_state=0)),
    ("lr", LogisticRegression(max_iter=1000)),
], voting="soft")
print("集成 CV-AUC =", round(cross_val_score(vote, X, y, cv=cv, scoring="roc_auc").mean(), 4))

In [ ]:
# ---- 最终模型 + 提交文件（Kaggle 格式）----
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
final = RandomForestClassifier(**rs.best_params_, random_state=0)
final.fit(X_train, y_train)

sub = pd.DataFrame({
    "PassengerId": X_test.index,
    "Survived": final.predict(X_test),
})
sub.to_csv("submission.csv", index=False)
print("已生成 submission.csv：", sub.shape)
print(sub.head().to_string())

from sklearn.metrics import accuracy_score
print("\n模拟测试集准确率 =", round(accuracy_score(y_test, sub["Survived"]), 4))
print("（真实 Kaggle：把 submission.csv 上传即可，格式完全一致）")

In [ ]:
# ---- 提交校验 + 特征重要性 + 复盘 ----
assert sub["Survived"].isin([0, 1]).all() and sub["Survived"].notna().all()
assert len(sub) == len(X_test)
print("提交文件校验通过：列 =", list(sub.columns), " 行 =", len(sub))

imp = pd.Series(final.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n特征重要性 Top8：")
print(imp.head(8).round(4).to_string())

# 复盘：错误分析驱动下一轮迭代（05 课闭环）
prob = final.predict_proba(X_test)[:, 1]
wrong = X_test[(prob > 0.5).astype(int).values != y_test.values]
print("\n错例数：", len(wrong), "/", len(X_test))
print("复盘问题：错例集中在哪个 Title / 舱位？缺什么特征？数据够吗？→ 回到特征工程")

## 课后练习（Kaggle）

1. **Titanic 完整复现**（<https://www.kaggle.com/c/titanic>）：把本课流程在 Kaggle 上跑通，用完整 891 行训练、提交 418 行测试，目标 **公共榜 ≥ 0.79**（0.80+ 进入前 10%）。用本课特征 + 集成即可达到。
2. **House Prices 全流程**（<https://www.kaggle.com/c/house-prices-advanced-regression-techniques>）：回归版完整工作流——EDA（缺失/偏态/离群）→ 对数变换目标 → 随机森林/GBDT → 调参 → 提交，目标 **RMSE < 0.14**（Top 25% 线）。
3. **自我复盘模板**：每次竞赛结束写三段话——①本次最大的特征工程发现；②交叉验证与公共榜分数的差异及原因；③下一轮迭代的第一优先级动作。
4. **下一站推荐**：进阶挑一个练——`Digit Recognizer`（图像入门）、`Porto Seguro`（不平衡+指标）、`Titanic` 用 XGBoost 复刻（学新工具）。